<style>
.custom-spacing {
    line-height: 0.5;
}
</style>

### <center>**TUGAS INDIVIDU**</center>
### <center>**KECERDASAN ARTIFISIAL UNTUK GEOFISIKA**</center>
#### <center>**Dosen Pengampu:** 
#### <center>**Dr. Muhammad Alimuddin Hamzah, M. Eng**</center>
#### <center>**Prof.  Dr. Samsu Arif, M. Si**</center>
#### <center>**Muhammad Farid Wajedy, S.Si, M.Si**</center>

<div style="text-align:center;">
  <img src="https://www.unhas.ac.id/wp-content/uploads/2022/09/Logo-Resmi-Unhas-1.png" alt="Deskripsi Gambar" style="width:280px;">
</div>

### <center>**DISUSUN OLEH :**</center>

#### <center>**HERMITA LESTARI           (H061241062)**<center>

### <center>**PROGRAM STUDI GEOFISIKA**</center>
### <center>**FAKULTAS MATEMATIKA DAN ILMU PENGETAHUAN ALAM**</center>
#### <center>**UNIVERSITAS HASANUDDIN**</center>
#### <center>**MAKASSAR**</center>
#### <center>**2026**</center>

<hr>

# Prompt AI untuk Earth Sciences (Multi-Topik)

Struktur mengikuti Bab 2 *AI Cookbook untuk Earth Sciences* (prompt sebagai *work specification*).


<div align="center">

| No | Bidang | Topik Prompt |
|----|--------|--------------|
| 1 | SIG | Audit layer SIG dan turunan terrain DEM untuk rencana delineasi DAS |
| 2 | Hidrometeorologi | QA/QC curah hujan harian dan agregasi bulanan |
| 3 | Geofisika padat | QC first-break seismik refraksi dangkal |
| 4 | Oseanografi | Audit profil CTD sebelum analisis massa air |
| 5 | Geodesi/Geofisika | Analisis time series GNSS untuk deformasi kerak |

</div>

<br>
<br>

<div align="center">

| Kode | Komponen | Fungsi |
|------|----------|--------|
| **R** | Role | Sudut pandang kerja, bukan credential |
| **T** | Task | Verb yang bisa diperiksa + objek + domain + kondisi penerimaan |
| **C** | Context | Tujuan, audiens, workflow, keputusan yang didukung |
| **I** | Input | Path, format, variabel, unit, CRS/datum, provenance |
| **K** | Constraints | File yang tidak boleh diubah, method, package, jaringan, approval |
| **O** | Output | Kontrak output: path, kolom/unit, format (rencana teks, belum hasil) |
| **Q** | Quality criteria | Kriteria yang dapat dinilai eksplisit |
| **V** | Optional verification | Check + expected + tindakan bila gagal |

</div>

## Template Umum

```markdown
## [R] Role
Bertindaklah sebagai <sudut pandang kerja>. Fokus pada <apa yang diperiksa>.

## [T] Task
<Verb> <objek> <domain/kondisi>, lalu <verb kedua>. <Kondisi penerimaan>.

## [C] Context
<Tujuan>, <audiens>, <tahap workflow>. Hasil dipakai untuk <keputusan>, bukan untuk <hal yang tidak dilakukan>.

## [I] Input
- `<path>`: format, variabel/kolom, unit, CRS/datum, sumber

## [K] Constraints
- Jangan mengubah <file/folder>.
- Method: <yang boleh>; <yang dilarang>.
- Software: <yang tersedia>; tanpa jaringan.
- Bila <kondisi tak jelas>: berhenti dan tanyakan.

## [O] Output (rencana teks)
- `<path output>`: <isi, kolom, unit>

## [Q] Quality criteria
- <kriteria terukur>

## [V] Optional verification
- Check: <command>. Expected: <kondisi>. Jika gagal: <tindakan>.
```

---

## Prompt 1 — SIG: Audit Layer dan Turunan Terrain DEM untuk Delineasi DAS

### [R] Role
Bertindaklah sebagai GIS data steward. Fokus pada preservasi Feature ID, coordinate order, CRS, metadata, dan validitas geometri sebelum layer dipakai untuk delineasi DAS.

### [T] Task
Periksa CRS, extent, resolusi, nodata, dan validitas geometri pada layer DEM, sungai, dan batas area studi, lalu hitung slope (derajat) dan aspect (derajat dari utara, searah jarum jam) dari DEM. Jangan melakukan delineasi DAS.

### [C]
Layer akan dipakai tim hidrologi untuk menyiapkan delineasi DAS di tahap berikutnya. Hasil audit dipakai untuk memutuskan apakah layer perlu diperbaiki sebelum overlay. Hasil ini bukan peta DAS final.

### [I] Input
- `data/raw/dem.tif`: GeoTIFF, elevasi dalam meter, CRS `{{EPSG:...}}`, resolusi `{{m}}`, nodata `{{-9999}}`, datum vertikal `{{MSL/tidak diketahui}}`.
- `data/raw/rivers.geojson`: LineString, atribut `river_id`, `name`, CRS `{{EPSG:4326}}`.
- `data/raw/study_area.geojson`: Polygon, atribut `area_id`, CRS `{{EPSG:...}}`.
- CRS target untuk analisis: `{{EPSG:...}}`.

### [K] Constraints
- Jangan mengubah `data/raw/`; jangan memperbaiki geometri di file asli.
- Jangan resampling DEM, mengisi sink, atau mengisi void.
- Lakukan reproyeksi hanya jika CRS sumber dan CRS target sama-sama dinyatakan. Jika CRS hilang, geometri tidak valid, atau coordinate order meragukan, hentikan prosedur dan tulis exception report.
- Jangan memperbaiki topologi (mis. `buffer(0)`) tanpa persetujuan.
- Software: Python dengan rasterio, geopandas, numpy, dan shapely bila tersedia; tanpa jaringan.

### [O] Output (rencana teks)
- `results/gis/layer_inventory.csv`: `layer`, `format`, `crs`, `extent`, `resolution_m`, `n_features`, `n_invalid_geometry`, `nodata_pct`.
- `results/gis/slope_deg.tif` dan `results/gis/aspect_deg.tif`: grid dengan CRS, transform, dan nodata identik dengan DEM.
- `results/gis/qc_report.md`: temuan per layer, overlap extent antar layer, daftar Feature ID bermasalah.
- `results/gis/run_config.json`: metode slope/aspect, ukuran kernel, CRS, versi package.

### [Q] Quality criteria
- Feature ID dan jumlah fitur input tidak berubah.
- Slope berada pada 0–90°; aspect pada 0–360° atau nodata untuk sel datar.
- Nodata DEM tetap nodata pada output.
- Semua layer punya CRS yang terdokumentasi, dan extent yang tidak saling overlap ditandai.

### [V] Optional verification
- Check: buka ulang `slope_deg.tif` dan bandingkan shape, CRS, dan transform dengan `dem.tif`. Expected: identik. Jika gagal: tandai output tidak valid.
- Check: uji fungsi slope pada bidang miring sintetis dengan sudut 10°. Expected: hasil 10° ± 0,1°. Jika gagal: laporkan bug di perhitungan.
- Check: SHA-256 semua file input sebelum dan sesudah. Expected: identik.

---

## Prompt 2 — Hidrometeorologi: QA/QC Curah Hujan Harian dan Agregasi Bulanan

### [R] Role
Bertindaklah sebagai reviewer QA/QC data hidrometeorologi. Fokus pada missing value, timestamp duplikat, unit, dan konsistensi zona waktu sebelum data dihitung.

### [T] Task
Normalisasikan `data/raw/rainfall-stations.csv` menjadi timestamp UTC dengan nama kolom konsisten, lalu hitung total curah hujan bulanan per `station_id` untuk Januari–Desember 2025 dalam mm. Simpan tabel dan ringkasan missing value beserta daftar record yang gagal diparse.

### [C]
Data 15 stasiun hujan harian diperiksa sebelum menghitung indeks kekeringan oleh tim lain. Output hanya dipakai untuk menemukan missing value, timestamp duplikat, dan unit yang tidak konsisten. Tahap ini bukan analisis klimatologi dan bukan prediksi.

### [I] Input
- `data/raw/rainfall-stations.csv`: kolom `station_id`, `timestamp`, `rainfall_mm`, `qc_flag_source` (bila ada). Zona waktu asal: `{{WIB/UTC/tidak diketahui}}`. Unit: mm per hari. Sumber: `{{BMKG/BWS/lainnya}}`.
- Periode: 2025-01-01 sampai 2025-12-31.

### [K] Constraints
- Jangan mengubah `data/raw/`; jangan menghapus record sumber.
- Jangan mengisi missing value dengan interpolasi, rata-rata, atau data satelit.
- Jika zona waktu asal tidak diketahui, jangan menebak: berhenti dan minta konfirmasi.
- Jangan mencampur label sumber stasiun, satelit, dan grid dalam satu series.
- Software: Python standard library dan pandas; tanpa jaringan.
- Jika ada nilai negatif atau di atas `{{500}}` mm/hari, beri flag, jangan diperbaiki.

### [O] Output (rencana teks)
- `results/qc/rainfall-normalized.csv`: `station_id`, `timestamp_utc`, `rainfall_mm`, `qc_flag`.
- `results/qc/rainfall-monthly.csv`: `station_id`, `month`, `total_mm`, `n_obs`, `n_missing`, `completeness_pct`.
- `results/qc/rainfall-summary.md`: jumlah stasiun, rentang waktu, total missing, duplikat, dan daftar flag.
- `results/qc/rejected.csv`: record gagal parse beserta alasan.

### [Q] Quality criteria
- Jumlah `station_id` unik dan jumlah record pada output sama dengan input (record gagal parse tetap muncul di `rejected.csv`).
- Bulan yang tidak lengkap punya `completeness_pct` yang dilaporkan, total tidak dipresentasikan sebagai bulan penuh.
- Satuan mm eksplisit pada semua kolom.
- Semua flag memuat alasan.

### [V] Optional verification
- Check: SHA-256 `rainfall-stations.csv` sebelum dan sesudah, serta jumlah baris dan daftar `station_id`. Expected: identik. Jika gagal: hentikan promosi output.
- Check: hitung ulang total bulanan tiga stasiun terpilih dengan script terpisah. Expected: sama dengan `rainfall-monthly.csv`. Jika gagal: laporkan selisih dan sumbernya.

---

## Prompt 3 — Geofisika Padat: QC First-Break Seismik Refraksi Dangkal

### [R] Role
Bertindaklah sebagai reviewer QC akuisisi seismik near-surface. Fokus pada geometri source-receiver, konsistensi picking, dan reciprocal time.

### [T] Task
Periksa tabel first-break `data/raw/refraction_picks.csv`, hitung offset untuk setiap pasangan shot–receiver, dan tandai pick dengan offset negatif, waktu tidak monoton terhadap offset, atau ketidakcocokan reciprocal time. Tulis QC report. Jangan menjalankan inversi.

### [C]
Survei refraksi dangkal untuk memetakan kedalaman batuan dasar pada lokasi rencana fondasi. QC dilakukan sebelum data diserahkan ke interpreter. Tahap ini tidak menentukan kecepatan lapisan atau kedalaman bidang bias.

### [I] Input
- `data/raw/refraction_picks.csv`: `shot_id`, `shot_x_m` (m), `receiver_id`, `receiver_x_m` (m), `t_fb_ms` (ms).
- Spasi geophone: `{{m}}`; jumlah geophone: `{{n}}`; sample interval rekaman: `{{ms}}`.
- Ambang reciprocal time: `{{ms}}`.

### [K] Constraints
- Jangan mengubah file picks; jangan menghapus baris.
- Jangan menjalankan inversi, membuat model kecepatan, atau memilih crossover point.
- Jangan membuat picking ulang dari trace; hanya periksa pick yang diberikan.
- Jika posisi geophone tidak konsisten dengan spasi yang dinyatakan, tandai dan hentikan bagian itu.
- Software: Python, NumPy, pandas; tanpa jaringan.

### [O] Output (rencana teks)
- `results/refraction/picks_qc.csv`: `shot_id`, `receiver_id`, `offset_m`, `t_fb_ms`, `flag`, `note`.
- `results/refraction/reciprocal_check.csv`: pasangan shot–receiver, `t_forward_ms`, `t_reverse_ms`, `diff_ms`, `status`.
- `results/refraction/qc_report.md`: jumlah pick, jumlah flag per jenis, shot tanpa pasangan reciprocal.
- `results/refraction/summary.json`: `n_picks`, `n_flagged`, `n_reciprocal_fail`.

### [Q] Quality criteria
- Jumlah baris `picks_qc.csv` sama dengan input.
- Setiap flag menyebut nilai, ambang, dan alasan.
- Offset dan waktu punya unit eksplisit.
- Pick yang tidak bisa diperiksa dipisahkan dari yang gagal QC.

### [V] Optional verification
- Check: hitung ulang offset untuk 10 baris acak. Expected: sama dengan `picks_qc.csv`. Jika gagal: periksa urutan kolom dan unit.
- Check: bandingkan jumlah pasangan reciprocal terhitung dengan jumlah shot × receiver yang seharusnya ada. Expected: selisih dilaporkan lengkap.

---

## Prompt 4 — Oseanografi: Audit Profil CTD Sebelum Analisis Massa Air

### [R] Role
Bertindaklah sebagai reviewer QC data oseanografi fisik. Fokus pada tekanan, temperatur, salinitas, waktu, dan lokasi profil CTD.

### [T] Task
Audit profil CTD `data/raw/ctd_cast_001.csv` untuk menemukan depth reversal, nilai hilang, spike, dan nilai di luar rentang fisik yang dinyatakan. Tulis daftar flag dan laporan QC tanpa mengubah observasi sumber.

### [C]
Profil dari satu cruise diperiksa sebelum dipakai pada analisis massa air. Output hanya berupa QC report dan tabel flag. Tahap ini tidak memakai klimatologi eksternal dan tidak menginterpretasikan massa air atau sirkulasi.

### [I] Input
- `data/raw/ctd_cast_001.csv`: `pressure_dbar`, `temperature_c` (ITS-90), `salinity_psu` (PSS-78), `time_utc`, `latitude`, `longitude`. Arah cast: `{{downcast/upcast}}`.
- Rentang fisik yang dipakai: temperatur `{{min–max °C}}`, salinitas `{{min–max}}`, spike threshold `{{nilai}}`.

### [K] Constraints
- Jangan mengubah `data/raw/`; simpan pressure, time, location, dan urutan observasi pada output audit.
- Jangan mengisi salinitas atau temperatur yang hilang; jangan menghapus quality flag sumber.
- Jangan memakai klimatologi eksternal atau menafsirkan massa air.
- Jika koordinat atau `time_utc` tidak konsisten dalam satu profil, tandai dan berhenti pada langkah yang bergantung padanya.
- Software: Python + pandas; tanpa jaringan.

### [O] Output (rencana teks)
- `results/ctd/ctd_cast_001_qc.csv`: seluruh kolom input + `qc_flag` + `qc_note`.
- `results/ctd/ctd_cast_001_flags.csv`: baris bermasalah dengan jenis flag.
- `results/ctd/qc_report.md`: jumlah observasi, depth reversal, missing, spike, out-of-range.
- `results/ctd/summary.json`: hitungan flag per jenis.

### [Q] Quality criteria
- Jumlah baris output = jumlah baris input; urutan observasi tidak berubah.
- Kode flag didefinisikan di laporan.
- Semua ambang dan rentang fisik tertulis eksplisit.
- Tidak ada nilai yang dimodifikasi.

### [V] Optional verification
- Check: implementasi ulang deteksi spike secara independen dengan ambang yang sama. Expected: himpunan flag sama. Jika gagal: laporkan selisihnya.
- Check: SHA-256 file input sebelum dan sesudah. Expected: identik.

---

## Prompt 5 — Pilihan: Analisis Time Series GNSS untuk Deformasi Kerak

### [R] Role
Bertindaklah sebagai reviewer analisis time series geodetik. Fokus pada kualitas data, model deterministik, dan pemisahan hasil terukur dari interpretasi.

### [T] Task
Untuk stasiun GNSS `{{STATION}}`, fit model linear ditambah komponen tahunan dan semi-tahunan pada komponen East, North, dan Up, lalu laporkan kecepatan (mm/tahun) dengan ketidakpastiannya, RMS residual, dan flag epoch outlier. Tandai kandidat offset pada tanggal di `data/raw/events.csv` tanpa mengoreksinya.

### [C]
Time series dipakai untuk tinjauan cepat laju deformasi pada zona tektonik/vulkanik aktif sebelum analisis lanjutan. Hasil bukan interpretasi sumber deformasi, bukan estimasi hazard, dan bukan koreksi offset final.

### [I] Input
- `data/raw/gnss/{{STATION}}.csv`: `date` (UTC), `east_mm`, `north_mm`, `up_mm`, `sig_e_mm`, `sig_n_mm`, `sig_u_mm`. Reference frame `{{IGS14/ITRF2014/lainnya}}`, sumber solusi `{{JPL/NGL/...}}`.
- `data/raw/events.csv`: `event_id`, `date`, `type` (gempa/penggantian antena), `note`.
- Ambang outlier: `{{n}}` × MAD.

### [K] Constraints
- Jangan mengubah `data/raw/`; jangan menghapus epoch, hanya beri flag.
- Gunakan hanya model yang dinyatakan (linear + tahunan + semi-tahunan); jangan menambah komponen post-seismic atau offset otomatis.
- Jangan mencampur reference frame yang berbeda.
- Jika gap lebih dari `{{n}}` hari, laporkan dan tetap gunakan seluruh data yang ada tanpa interpolasi.
- Software: Python, NumPy, SciPy, pandas yang tersedia; tanpa jaringan.

### [O] Output (rencana teks)
- `results/gnss/{{STATION}}_fit.csv`: `component`, `velocity_mm_yr`, `sigma_velocity`, `annual_amp_mm`, `semiannual_amp_mm`, `rms_mm`, `n_epochs`.
- `results/gnss/{{STATION}}_residuals.csv`: `date`, `res_e`, `res_n`, `res_u`, `flag`.
- `results/gnss/{{STATION}}_flags.csv`: outlier, gap, dan kandidat offset dengan tanggal serta alasannya.
- `results/gnss/qc_report.md` dan `run_config.json`: parameter model, ambang, versi package.

### [Q] Quality criteria
- Kecepatan selalu dilaporkan bersama ketidakpastian dan periode data.
- Jumlah epoch input = jumlah epoch pada residual.
- Semua ambang dan parameter model tercatat di `run_config.json`.
- Laporan memisahkan hasil terukur dari hal yang belum ditafsirkan.

### [V] Optional verification
- Check: fit pada series sintetis dengan kecepatan 5 mm/tahun dan noise diketahui. Expected: kecepatan terpulihkan dalam `{{toleransi}}` mm/tahun. Jika gagal: laporkan bug pada model.
- Check: rata-rata residual per komponen. Expected: mendekati nol (`{{toleransi}}` mm). Jika tidak: periksa implementasi fit.
- Check: SHA-256 file input sebelum dan sesudah. Expected: identik.

---

## Cara Pakai
1. Pilih satu prompt dan isi semua `{{...}}` serta path file.
2. Jalankan, lalu nilai hasil aktual terhadap **[Q]** dan **[V]**.
3. Bila prompt terlalu panjang, pindahkan detail ke file configuration/recipe pendamping dan pertahankan **Constraints** serta **Quality criteria** di prompt.
4. Simpan diff dan `run_config.json` sebagai bukti perubahan.